In [1]:
import os
import sys
import glob

import joblib
import pandas as pd
import numpy as np
from pathlib import Path
 
sys.path.insert(0, '../../..')
from utils.constants import Configs
from utils.bootstrap_utils import paired_subject_bootstrap
 
RESULTS_DIR = Configs._wrist_only_loso_results_cache_dir

In [2]:
def load_all_oof_movement(results_dir):
    """
    Load the OOF_Movement (long-format: sub, repID, movID, true_is_outlier,
    pred_is_outlier, anomaly_score) data from all .pkl files, with
    Model/Representation/Phase.
    """
    rows = []
    for file in glob.glob(os.path.join(results_dir, "*.pkl")):
        obj = joblib.load(file)
        if not (isinstance(obj, dict) and "OOF_Movement" in obj):
            continue
 
        oof = obj["OOF_Movement"].copy()
        stem = Path(file).stem.replace("wrist_only_loso_results_", "")
        model, representation, phase = stem.rsplit("_", 2)
        oof["Model"] = model
        oof["Representation"] = representation
        oof["Phase"] = phase
        rows.append(oof)
 
    if not rows:
        raise RuntimeError(f"Nenhum ficheiro com 'OOF_Movement' encontrado em {results_dir}.")
    return pd.concat(rows, ignore_index=True)

In [3]:
oof_data = load_all_oof_movement(RESULTS_DIR)
print(f"Loaded {len(oof_data)} movement-level OOF rows.")
oof_data.head()

Loaded 5225 movement-level OOF rows.


,sub,repID,movID,pred_is_outlier,true_is_outlier,anomaly_score,Model,Representation,Phase,movLabel
0,1,1,9,0,0,0.056705,ZScore,ShapeBased,Move,NaN
1,1,2,9,0,0,0.052823,ZScore,ShapeBased,Move,NaN
2,1,3,9,0,0,0.372943,ZScore,ShapeBased,Move,NaN
3,1,4,9,0,0,0.012485,ZScore,ShapeBased,Move,NaN
4,1,5,9,0,0,0.059313,ZScore,ShapeBased,Move,NaN


In [4]:
def _pair_stats(oof, pair_a, pair_b, phase, metric, n_boot, seed):
    """
    Compute the bootstrap statistics for a pair of (model, representation) tuples.
    """
    sel = lambda m, r: oof[(oof['Model'] == m) & (oof['Representation'] == r) & (oof['Phase'] == phase)]
    a, b = sel(*pair_a), sel(*pair_b)
    if a.empty or b.empty:
        return None
    obs, d, p, pa, pb, n_common = paired_subject_bootstrap(a, b, metric, n_boot, seed)
    if len(d) == 0:
        return None
    return dict(obs=obs, d=d, p=p, pa=pa, pb=pb, n_common=n_common, n_valid=len(d),
                lo=np.percentile(d, 2.5), hi=np.percentile(d, 97.5))

def _format_p_value(p, n_boot):
    floor = 1.0 / n_boot
    return f"<{floor:.4f}" if p <= floor else f"{p:.4f}"

In [5]:
def run_pairwise_comparisons(oof_data, phases, pairs, metric='mcc', n_boot=2000,
                             seed=42, output_csv=None):
    """
    Paired subject-level bootstrap for a list of ((model, rep), (model, rep)) pairs.
    Bonferroni: alpha = 0.05 / k, with k = valid comparisons in each phase.
    The difference shown is the OBSERVED A - B; the CI comes from the bootstrap.
    """
    rows = []
    for phase in phases:
        stats = []

        # Compute the bootstrap statistics for each pair of (model, representation) tuples
        for pair_a, pair_b in pairs:
            s = _pair_stats(oof_data, pair_a, pair_b, phase, metric, n_boot, seed)
            if s is None:
                print(f"No valid data: {pair_a} vs {pair_b} ({phase})")
                continue
            stats.append((pair_a, pair_b, s))
        if not stats:
            continue

        # Bonferroni correction: alpha = 0.05 / k
        k = len(stats)
        alpha = 0.05 / k

        # Determine significance and which model is better for each pair
        for pair_a, pair_b, s in stats:
            sig_unc = s['p'] < 0.05  # Uncorrected significance
            sig_cor = s['p'] < alpha # Bonferroni correction

            # Determine which model is better based on the observed difference
            if sig_cor:
                better = pair_a if s['obs'] > 0 else pair_b
            else:
                better = "ns"
            rows.append({
                'Phase': phase,
                'A': f"{pair_a[0]} ({pair_a[1]})",
                'B': f"{pair_b[0]} ({pair_b[1]})",
                f'{metric.upper()}_A': round(s['pa'], 2),
                f'{metric.upper()}_B': round(s['pb'], 2),
                'Diff (A-B)': f"{s['obs']:+.2f}",
                'CI 95%': f"[{s['lo']:+.2f}, {s['hi']:+.2f}]",
                'p': _format_p_value(s['p'], n_boot),
                'alpha': f"{alpha:.4f} (k={k})",
                'Sig. uncorrected': sig_unc,
                'Sig. corrected': sig_cor,
                'Better': better if better == "ns" else f"{better[0]} ({better[1]})",
                'N subjects': s['n_common'],
                'Valid resamples': s['n_valid'],
            })

    out = pd.DataFrame(rows)
    if output_csv:
        out.to_csv(output_csv, index=False)
    return out

## Each detector against the Z-Score baseline (k per representation and phase)

In [6]:
models = ["IForest", "DBSCAN", "LOF", "OCSVM", "MatrixProfile"]
tables = []

# Build pairs for each representation type 
for rep in ['ShapeBased', 'FeatureBased']:
    pairs = [((m, rep), ("ZScore", rep)) for m in models]     # MatrixProfile has no FeatureBased: skipped, k is correct
    t = run_pairwise_comparisons(oof_data, ['Pick', 'Move'], pairs, metric='mcc')
    t.insert(0, 'Representation', rep)
    tables.append(t)

baseline_df = pd.concat(tables, ignore_index=True)
baseline_df.to_csv("bootstrap_vs_baseline_mcc.csv", index=False)
baseline_df

No valid data: ('MatrixProfile', 'FeatureBased') vs ('ZScore', 'FeatureBased') (Pick)
No valid data: ('MatrixProfile', 'FeatureBased') vs ('ZScore', 'FeatureBased') (Move)


,Representation,Phase,A,B,MCC_A,MCC_B,Diff (A-B),CI 95%,p,alpha,Sig. uncorrected,Sig. corrected,Better,N subjects,Valid resamples
0,ShapeBased,Pick,IForest (ShapeBased),ZScore (ShapeBased),0.13,0.11,+0.02,"[-0.06, +0.16]",0.8370,0.0100 (k=5),False,False,ns,19,2000
1,ShapeBased,Pick,DBSCAN (ShapeBased),ZScore (ShapeBased),0.33,0.11,+0.22,"[+0.01, +0.43]",0.0450,0.0100 (k=5),True,False,ns,19,2000
2,ShapeBased,Pick,LOF (ShapeBased),ZScore (ShapeBased),0.27,0.11,+0.16,"[-0.12, +0.47]",0.2440,0.0100 (k=5),False,False,ns,19,2000
3,ShapeBased,Pick,OCSVM (ShapeBased),ZScore (ShapeBased),0.23,0.11,+0.12,"[-0.08, +0.32]",0.2080,0.0100 (k=5),False,False,ns,19,2000
4,ShapeBased,Pick,MatrixProfile (ShapeBased),ZScore (ShapeBased),0.27,0.11,+0.16,"[-0.03, +0.32]",0.0840,0.0100 (k=5),False,False,ns,19,2000
5,ShapeBased,Move,IForest (ShapeBased),ZScore (ShapeBased),0.50,0.65,-0.16,"[-0.64, +0.00]",0.7194,0.0100 (k=5),False,False,ns,19,1996
6,ShapeBased,Move,DBSCAN (ShapeBased),ZScore (ShapeBased),0.62,0.65,-0.03,"[-0.41, +0.48]",0.9238,0.0100 (k=5),False,False,ns,19,1996
7,ShapeBased,Move,LOF (ShapeBased),ZScore (ShapeBased),0.29,0.65,-0.37,"[-0.77, +0.21]",0.1623,0.0100 (k=5),False,False,ns,19,1996
8,ShapeBased,Move,OCSVM (ShapeBased),ZScore (ShapeBased),0.22,0.65,-0.44,"[-0.73, +0.02]",0.0601,0.0100 (k=5),False,False,ns,19,1996
9,ShapeBased,Move,MatrixProfile (ShapeBased),ZScore (ShapeBased),0.23,0.65,-0.43,"[-0.82, +0.08]",0.0802,0.0100 (k=5),False,False,ns,19,1996


## Direct Comparisons


In [7]:
# Define Direct Pairs (Config A vs Config B):
direct_pairs = [
    # 1. Compare DBSCAN vs IForest withing each representation
    (("IForest", "FeatureBased"), ("DBSCAN", "FeatureBased")),
    (("IForest", "ShapeBased"),   ("DBSCAN", "ShapeBased")),

    # 2. Compare FeatureBased vs ShapeBased (DTW) for the same algorithm
    (("ZScore",  "FeatureBased"), ("ZScore",  "ShapeBased")),
    (("DBSCAN",  "FeatureBased"), ("DBSCAN",  "ShapeBased")),
    (("IForest", "FeatureBased"), ("IForest", "ShapeBased")),
    (("LOF",     "FeatureBased"), ("LOF",     "ShapeBased")),
    (("OCSVM",     "FeatureBased"), ("OCSVM",     "ShapeBased")),
]

direct_df = run_pairwise_comparisons(oof_data, ['Pick', 'Move'], direct_pairs,
                                     metric='mcc', output_csv="bootstrap_direct_mcc.csv")
direct_df

,Phase,A,B,MCC_A,MCC_B,Diff (A-B),CI 95%,p,alpha,Sig. uncorrected,Sig. corrected,Better,N subjects,Valid resamples
0,Pick,IForest (FeatureBased),DBSCAN (FeatureBased),0.50,0.54,-0.04,"[-0.21, +0.08]",0.5620,0.0071 (k=7),False,False,ns,19,2000
1,Pick,IForest (ShapeBased),DBSCAN (ShapeBased),0.13,0.33,-0.20,"[-0.41, +0.04]",0.0800,0.0071 (k=7),False,False,ns,19,2000
2,Pick,ZScore (FeatureBased),ZScore (ShapeBased),0.25,0.11,+0.13,"[-0.06, +0.30]",0.1610,0.0071 (k=7),False,False,ns,19,2000
3,Pick,DBSCAN (FeatureBased),DBSCAN (ShapeBased),0.54,0.33,+0.21,"[+0.04, +0.38]",0.0160,0.0071 (k=7),True,False,ns,19,2000
4,Pick,IForest (FeatureBased),IForest (ShapeBased),0.50,0.13,+0.37,"[+0.02, +0.66]",0.0390,0.0071 (k=7),True,False,ns,19,2000
5,Pick,LOF (FeatureBased),LOF (ShapeBased),0.24,0.27,-0.03,"[-0.33, +0.35]",0.8640,0.0071 (k=7),False,False,ns,19,2000
6,Pick,OCSVM (FeatureBased),OCSVM (ShapeBased),0.42,0.23,+0.19,"[+0.06, +0.33]",0.0030,0.0071 (k=7),True,True,OCSVM (FeatureBased),19,2000
7,Move,IForest (FeatureBased),DBSCAN (FeatureBased),1.00,0.57,+0.43,"[+0.16, +0.74]",0.0060,0.0071 (k=7),True,True,IForest (FeatureBased),19,1996
8,Move,IForest (ShapeBased),DBSCAN (ShapeBased),0.50,0.62,-0.12,"[-0.70, +0.31]",0.5110,0.0071 (k=7),False,False,ns,19,1996
9,Move,ZScore (FeatureBased),ZScore (ShapeBased),0.65,0.65,-0.00,"[-0.44, +0.58]",0.9399,0.0071 (k=7),False,False,ns,19,1996
